# SDXL NPU Conversion
This notebook mounts Google Drive, downloads the Qualcomm QNN SDK, and runs the SDXL conversion pipeline.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title Empty Google Drive Trash (Optional)
#@markdown Run this cell if you want to permanently empty your Google Drive trash to reclaim space.
#@markdown Note: This will require you to authenticate your Google Account again specifically for the Drive API.

empty_gdrive_trash = False #@param {type:"boolean"}

if empty_gdrive_trash:
    from google.colab import auth
    from googleapiclient.discovery import build

    print("Authenticating to access Google Drive API...")
    auth.authenticate_user()

    print("Emptying Google Drive trash...")
    drive_service = build("drive", "v3")
    drive_service.files().emptyTrash().execute()
    print("Google Drive trash emptied successfully!")
else:
    print("Skipped emptying trash.")


Skipped emptying trash.


In [3]:
### @title 1. Setup Environment (Multi-Backend & Auto-Archive)
setup_mode = "Install" #@param ["Extract", "Install"]
compute_type = "cpu" #@param ["gpu", "cpu"]
auto_archive = False #@param {type:"boolean"}
import os
import sys
import shutil

# Prevent __pycache__ warnings during archiving
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

# --- CONFIGURATION ---
drive_base = "/content/drive/MyDrive/SDXL"
drive_repo = f"{drive_base}/convertsdxl"
local_lib_path = "/content/local_lib"
local_archive_temp = f"/content/google_lib_{compute_type}.tar.gz"
drive_lib_archive = f"{drive_repo}/google_lib_{compute_type}.tar.gz"
drive_modules_archive = f"{drive_repo}/redefined_modules_backup.tar.gz"

os.makedirs(drive_base, exist_ok=True)
%cd {drive_base}

if not os.path.exists(drive_repo):
    !git clone https://github.com/Lycra-GX/convertsdxl.git --depth=1

%cd {drive_repo}
os.makedirs(local_lib_path, exist_ok=True)

# --- GENERATE PYPROJECT.TOML ---
if compute_type == "gpu":
    toml_content = """[project]\nname = \"qnn\"\nversion = \"0.1.0\"\ndependencies = [\n    \"torch==2.5.1\", \"accelerate\", \"diffusers==0.31.0\", \"transformers==4.46.1\",\n    \"numpy==1.26.4\", \"onnx>=1.18.0\", \"packaging>=25.0\", \"pandas>=2.2.3\",\n    \"pip>=25.1.1\", \"pyyaml>=6.0.2\"\n]"""
else:
    toml_content = """[project]\nname = \"qnn\"\nversion = \"0.1.0\"\ndependencies = [\n    \"torch==2.5.1+cpu\", \"accelerate\", \"diffusers==0.31.0\", \"transformers==4.46.1\",\n    \"numpy==1.26.4\", \"onnx>=1.18.0\", \"packaging>=25.0\", \"pandas>=2.2.3\",\n    \"pip>=25.1.1\", \"pyyaml>=6.0.2\"\n]\n[tool.uv.sources]\ntorch = [{ index = \"pytorch-cpu\" }]\ntorchvision = [{ index = \"pytorch-cpu\" }]\n[[tool.uv.index]]\nname = \"pytorch-cpu\"\nurl = \"https://download.pytorch.org/whl/cpu\"\nexplicit = true"""

with open("pyproject.toml", "w") as f:
    f.write(toml_content)

# --- INSTALL / EXTRACT ---
if setup_mode == "Install":
    print(f"🌐 Installing {compute_type.upper()} dependencies...")
    if shutil.which("uv") is None:
        !pip install uv -q

    index_url = "https://download.pytorch.org/whl/cu121" if compute_type == "gpu" else "https://download.pytorch.org/whl/cpu"
    !uv pip install --target {local_lib_path} --force-reinstall torch torchvision torchaudio --index-url {index_url}
    !uv pip install --target {local_lib_path} -r pyproject.toml

    # Sync redefined_modules into local_lib
    if os.path.exists("redefined_modules"):
        print("📦 Backing up and linking redefined_modules...")
        # Suppress pycache in tar to avoid 'file changed as we read it' warnings
        !tar --exclude='*/__pycache__*' -czf {drive_modules_archive} redefined_modules
        shutil.copytree("redefined_modules", os.path.join(local_lib_path, "redefined_modules"), dirs_exist_ok=True)

    if auto_archive:
        print(f"📦 Compressing locally to {local_archive_temp}...")
        !tar --exclude='*/__pycache__*' -czf {local_archive_temp} -C {local_lib_path} .
        print(f"💾 Moving archive to Drive: {drive_lib_archive}...")
        shutil.move(local_archive_temp, drive_lib_archive)

if setup_mode == "Extract":
    if os.path.exists(drive_lib_archive):
        print(f"📦 Extracting {compute_type.upper()} archive...")
        !tar -xzf {drive_lib_archive} -C {local_lib_path}
    if os.path.exists(drive_modules_archive):
        print("📦 Extracting redefined_modules into local_lib...")
        !tar -xzf {drive_modules_archive} -C {local_lib_path}

print(f"\n✅ Finished. Please RESTART SESSION (Runtime > Restart session) if changing backends.")

/content/drive/MyDrive/SDXL
/content/drive/MyDrive/SDXL/convertsdxl
🌐 Installing CPU dependencies...
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Resolved 14 packages in 1.68s
Prepared 14 packages in 9.12s
Installed 14 packages in 287ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.4
 + pillow==12.2.0
 + setuptools==70.2.0
 + sympy==1.14.0
 + torch==2.12.1+cpu
 + torchaudio==2.11.0+cpu
 + torchvision==0.27.1+cpu
 + typing-extensions==4.15.0
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Resolved 38 packages in 1.24s
Prepared 29 packages in 20.52s
Uninstalled 3 packages in 507ms
Installed 29 packages in 693ms
 + accelerate==1.14.0
 + certifi==2026.6.17
 + charset-normalizer==3.4.7
 + diffusers==0.31.0
 + hf-xet==1.5.1
 + huggingface-hub==0.36.2
 + idna==3.18
 + importlib-metadata==9.0.0
 + ml-dtypes==0.5.4
 - numpy==2.4.4
 + numpy==1.26.4
 + onnx==1.22.0
 + packaging==26.2
 + pandas==3.0.

In [4]:
import os

local_lib_path = "/content/local_lib"

if os.path.exists(local_lib_path):
    files = sorted(os.listdir(local_lib_path))
    print(f"✅ Local cache found at {local_lib_path}")
    print(f"Total items: {len(files)}")
    print("Full directory contents:")
    # Print items with line breaks for better readability
    for item in files:
        print(item)
else:
    print(f"❌ Local cache directory {local_lib_path} does not exist yet.")

✅ Local cache found at /content/local_lib
Total items: 95
Full directory contents:
.lock
81d243bd2c585b0f4821__mypyc.cpython-312-x86_64-linux-gnu.so
PIL
_distutils_hack
_yaml
accelerate
accelerate-1.14.0.dist-info
bin
certifi
certifi-2026.6.17.dist-info
charset_normalizer
charset_normalizer-3.4.7.dist-info
dateutil
diffusers
diffusers-0.31.0.dist-info
distutils-precedence.pth
filelock
filelock-3.29.0.dist-info
fsspec
fsspec-2026.4.0.dist-info
functorch
google
hf_xet
hf_xet-1.5.1.dist-info
huggingface_hub
huggingface_hub-0.36.2.dist-info
idna
idna-3.18.dist-info
importlib_metadata
importlib_metadata-9.0.0.dist-info
isympy.py
jinja2
jinja2-3.1.6.dist-info
markupsafe
markupsafe-3.0.3.dist-info
ml_dtypes
ml_dtypes-0.5.4.dist-info
mpmath
mpmath-1.3.0.dist-info
networkx
networkx-3.6.1.dist-info
numpy
numpy-1.26.4.dist-info
numpy.libs
onnx
onnx-1.22.0.dist-info
packaging
packaging-26.2.dist-info
pandas
pandas-3.0.3.dist-info
pillow-12.2.0.dist-info
pillow.libs
pip
pip-26.1.2.dist-info
pkg_res

In [5]:
#@title 2. Link Libraries (Optimized Local Link)
import sys
import os
import importlib

local_lib_path = "/content/local_lib"

# Ensure the local path is at the front of sys.path
if os.path.exists(local_lib_path):
    if local_lib_path not in sys.path:
        sys.path.insert(0, local_lib_path)

    os.environ['PYTHONPATH'] = local_lib_path + ":" + os.environ.get('PYTHONPATH', '')
    print(f"✅ Local library path registered: {local_lib_path}")

    try:
        import torch
        # If torch is already loaded but from a different path, we might need to reload or verify
        if "local_lib" not in torch.__file__:
            print("⚠️ Torch was loaded from a different path. Attempting to switch...")
            for module in list(sys.modules):
                if module.startswith("torch"):
                    del sys.modules[module]
            import torch

        print(f"✅ Active Torch Path: {torch.__file__}")
        print(f"✅ Torch Version: {torch.__version__}")
    except Exception as e:
        print(f"❌ Import verification result: {e}")
        print("💡 Hint: If you see docstring errors, the environment is likely already set up correctly.")
else:
    print("❌ Local library folder not found. Please run the Setup cell first.")

✅ Local library path registered: /content/local_lib
✅ Active Torch Path: /content/local_lib/torch/__init__.py
✅ Torch Version: 2.5.1+cpu


In [6]:
#@title 3. Environment Verification
import os
import torch

print("--- Status Check ---")
print(f"✅ Torch Version: {torch.__version__}")
print(f"🔥 Persistent Libs: {'Yes' if 'drive/MyDrive' in torch.__file__ else 'No'}")

# Check for GPU
if torch.cuda.is_available():
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected. Check runtime type (Tesla T4).")

scripts = ["prepare_data_sdxl.py", "gen_quant_data_sdxl.py", "export_onnx_sdxl.py", "export_sdxl.sh"]
missing = [s for s in scripts if not os.path.exists(f"/content/drive/MyDrive/SDXL/convertsdxl/{s}")]

if not missing:
    print("✅ All pipeline scripts are present. Ready for conversion!")
else:
    print(f"❌ Missing files: {missing}")

--- Status Check ---
✅ Torch Version: 2.5.1+cpu
🔥 Persistent Libs: No
⚠️ No GPU detected. Check runtime type (Tesla T4).
✅ All pipeline scripts are present. Ready for conversion!


In [7]:
import os
import sys
from pathlib import Path

# Get current Colab python version
current_py = f"{sys.version_info.major}.{sys.version_info.minor}"

model_path = "/content/drive/MyDrive/SDXL/models/PVCStyleModelMovable_beta27Realistic.safetensors" #@param {type:"string"}
model_name = "Movable_figure_model" #@param {type:"string"}
realistic = True #@param {type:"boolean"}
scheduler = "dpm" #@param ["dpm", "lcm", "eulera"]
cfg = "5,7" #@param {type:"string"}
steps = "15,30" #@param {type:"string"}
soc_version = "min" #@param {type:"string"}
data_prep_device = "cpu" #@param ["gpu", "cpu"] {type:"string"}

local_lib_path = "/content/local_lib"
local_cache_path = "/content/huggingface_cache"
system_cache = "/root/.cache/huggingface"

# Synchronize Environment Variables
os.environ['HF_HOME'] = local_cache_path
os.environ['TRANSFORMERS_CACHE'] = local_cache_path
os.environ['HF_DATASETS_CACHE'] = local_cache_path

# Deep Suppression: Create version flags and .no_migration
for base_path in [local_cache_path, system_cache]:
    for sub in ["", "hub", "diffusers", "models", "datasets"]:
        p = Path(base_path) / sub
        p.mkdir(parents=True, exist_ok=True)
        for fname in ["version.txt", "version_diffusers_cache.txt", ".no_migration"]:
            with open(p / fname, "w") as f:
                f.write("4.22.0")

export_script_content = f"""set -e
# Fix: Added '.' to PYTHONPATH so redefined_modules can be found
export PYTHONPATH=.:{local_lib_path}:$PYTHONPATH
export HF_HOME={local_cache_path}
export TRANSFORMERS_CACHE={local_cache_path}
export HF_DATASETS_CACHE={local_cache_path}

echo 'ℹ️ Using persistent cache mode.'

source .venv/bin/activate || (uv venv -p {current_py} && source .venv/bin/activate)

realistic_flag=""
if [ \"{realistic}\" = \"True\" ]; then realistic_flag="--realistic"; fi

# Run conversion
python prepare_data_sdxl.py --model_path \"{model_path}\" $realistic_flag --scheduler {scheduler} --cfg {cfg} --step {steps}
python gen_quant_data_sdxl.py
python export_onnx_sdxl.py --model_path \"{model_path}\"

for soc in {soc_version}; do
    bash scripts/convert_all_sdxl.sh --min_soc $soc
done
"""

with open("/content/drive/MyDrive/SDXL/convertsdxl/export_sdxl.sh", "w") as f:
    f.write(export_script_content)

print(f"✅ export_sdxl.sh updated with local path fix. Please re-run the execution cell.")

✅ export_sdxl.sh updated with local path fix. Please re-run the execution cell.


In [ ]:
import os
# Check where Transformers thinks the cache is
from transformers.utils import TRANSFORMERS_CACHE
print(f"Target Path: {TRANSFORMERS_CACHE}")

paths_to_check = [TRANSFORMERS_CACHE, "/root/.cache/huggingface", "/root/.cache/huggingface/hub"]
for p in paths_to_check:
    vf = os.path.join(p, "version.txt")
    exists = os.path.exists(vf)
    content = open(vf).read().strip() if exists else "MISSING"
    print(f"{vf} -> {content}")

In [ ]:
### @title 4. Execute Conversion
%cd /content/drive/MyDrive/SDXL/convertsdxl
!bash export_sdxl.sh